# Simple maximum likelihood demo
Given measurements $y$ we wish to find the values of $\theta$ using the following model
$
 y_i = h(\theta)_i + e_i,
$
where
$
h(\theta)_i = a_i \theta + 0.5.
$
and
$
e_i \sim \mathcal{N}(0, \sigma^2)
$
Here, we assume the $a_i$'s are known but that we do not know the value of $\sigma$ and so we will use maximum likelihood to learn $\theta$ and $\sigma$

## step 1) simulate some measurements
Using the 'true' $\theta$ and $\sigma$ values we simulate some measurements

In [24]:
import numpy as np

N = 10                              # number of measurements
a = np.linspace(5,10,N)                              # parameter in measurement model
h = lambda theta: a*theta + 0.5     # measurement function


theta = 1.                          # true values
sigma = 0.2

y = h(theta) + sigma * np.random.randn(N)
print(y)

[ 5.23888682  6.24636751  6.75936121  7.20204066  7.8518966   8.36158982
  8.78360866  9.18425414  9.88599719 10.75623656]


## Step 2) Define the log likelihood
Letting $y = [y_1,...,y_N]$ and $h(\theta) = [h(\theta)_i,...,h(\theta)_N]$
Because we assumed gaussian noise the likelihood is given by
$
l(\theta,\sigma) =\frac{1}{\sqrt{|2*\pi\Sigma|}}\exp\left(0.5(y-h(\theta))^T\Sigma^{-1}(y-h(\theta))\right)
$
Where in our case $\Sigma$ is an $N$-by-$N$ matrix with diagonal elements $\sigma^2$ and zeros otherwise.

However, it is computational more robust to work with the log likelihood
$
logl(\theta,\sigma) = -0.5\log(|2*\pi\Sigma|) - 0.5(y-h(\theta))^T\Sigma^{-1}(y-h(\theta))
$
Since in this case we have indepedent measurements we can also further simplify this quite a bit
$
logl(\theta,\sigma) = \sum_{i=1}^{i=N}-0.5\log(\sigma\sqrt{2\pi}) - 0.5\frac{(y_i-h(\theta)_i)^2}{\sigma^2}
$

In [25]:
# define a function to compute the log likelihood
# note that this function is not written very efficiently and ther is also a lot of computational tricks that can be done, for instance if computing the log determinant of a covariance matrix one should use a cholesky factor to do it efficiently
def nlogl(vars):
    theta = vars[0]
    sigma = vars[1]
    ypred = h(theta)        # this uses the known a_i's
    tmp = 0.5*np.log(np.sqrt(2*np.pi*sigma**2)) + 0.5*(y-ypred)**2/sigma**2
    return tmp.sum()


## step 3) Solve the optimisation problem
Normaly we would also want to specify derivative information i.e. $ \frac{\partial logl}{\partial \theta} $ and $ \frac{\partial logl}{\partial \sigma} $, however, because this is a really simple problem we will just try to get away without it.

$\sigma$ should also be bounded below by 0

In [28]:
from scipy.optimize import minimize
var0 = np.array([0, 1.])    # initial guess for theta and sigma
res = minimize(nlogl, var0)

theta_est = res.x[0]
sigma_est = res.x[1]

print('true theta is ', theta, ' ML estimate is ', theta_est)
print('true sigma is ', sigma, ' ML estimate is ', sigma_est)

true theta is  1.0  ML estimate is  1.003846704950721
true sigma is  0.2  ML estimate is  0.22644185914437595
